# 05 — Fuseaux horaires et sessions

Ce notebook part du bug qui a motivé toute la politique de fuseaux de la bibliothèque, puis
montre l'outil qui le referme : `session_of`.

La règle tient en trois lignes :

1. **Naïf = une étiquette.** La partie date est prise au pied de la lettre, sans conversion.
2. **Aware = un instant.** Le projeter sur un jour calendaire exige un fuseau explicite.
3. **Les offsets préservent l'heure murale et le `tzinfo`.** Ils ne touchent que la date.

In [1]:
from datetime import date, datetime, timedelta, timezone
from zoneinfo import ZoneInfo

import pandas as pd

import better_calendar as bcal
from better_calendar import Calendar

## 1. Le bug

Le même instant. Deux réponses. Aucun avertissement.

In [2]:
ts = pd.Timestamp("2026-07-31 23:30", tz="UTC")

print("l'instant                :", ts)
print("ts.date()                :", ts.date(), f"({ts.date():%A})")
print("ts.tz_convert('Paris')   :", ts.tz_convert("Europe/Paris").date(),
      f"({ts.tz_convert('Europe/Paris').date():%A})")

l'instant                : 2026-07-31 23:30:00+00:00
ts.date()                : 2026-07-31 (Friday)
ts.tz_convert('Paris')   : 2026-08-01 (Saturday)


`.date()` répond en fonction du fuseau que le timestamp trimballe. Dans un pipeline où un
`tz_convert` traîne trois fonctions plus haut, la réponse change sans que personne ne
décide quoi que ce soit — et la conséquence n'est pas cosmétique :

In [3]:
utc = Calendar("utc", tz="UTC")
paris = Calendar("paris", tz="Europe/Paris")

print("jour ouvré en UTC   ?", utc.is_bday(ts))
print("jour ouvré à Paris  ?", paris.is_bday(ts))

jour ouvré en UTC   ? True
jour ouvré à Paris  ? False


## 2. La parade

`session_of` oblige à nommer le référentiel, et le dit en retour.

In [4]:
pd.DataFrame(
    [
        {"référentiel": nom, "jour": bcal.session_of(ts, tz=zone),
         "nom du jour": bcal.session_of(ts, tz=zone).strftime("%A")}
        for nom, zone in [
            ("UTC", "UTC"), ("Europe/Paris", "Europe/Paris"),
            ("Asia/Tokyo", "Asia/Tokyo"), ("America/New_York", "America/New_York"),
        ]
    ]
).set_index("référentiel")

,jour,nom du jour
référentiel,,
UTC,2026-07-31,Friday
Europe/Paris,2026-08-01,Saturday
Asia/Tokyo,2026-08-01,Saturday
America/New_York,2026-07-31,Friday


Et il n'y a **pas** de repli silencieux. Le calendrier `weekday` ne déclare aucun fuseau :

In [5]:
try:
    bcal.session_of(ts)
except bcal.AmbiguousTimezoneError as exc:
    print(exc)

session_of needs a timezone: calendar 'weekday' declares none, so there is no frame to read an instant in. Pass tz=..., use a calendar that has one, or set better_calendar.config.default_tz. A composite calendar loses its timezone when its operands disagree, which is usually the cause.


Trois façons de le fournir : l'argument, le calendrier, ou l'échappatoire globale.

In [6]:
print("via l'argument  :", bcal.session_of(ts, tz="Europe/Paris"))
print("via le calendrier:", bcal.session_of(ts, cal=paris))

bcal.config.default_tz = "Europe/Paris"        # la seule variable globale de la bibliothèque
print("via le défaut   :", bcal.session_of(ts))
bcal.config.default_tz = None

via l'argument  : 2026-08-01
via le calendrier: 2026-08-01
via le défaut   : 2026-08-01


## 3. Naïf contre aware

Un `datetime` naïf est une **étiquette**, pas un instant : sa partie date est lue telle
quelle, sans conversion, même si on passe un fuseau.

In [7]:
naif = datetime(2026, 7, 31, 23, 30)
aware = datetime(2026, 7, 31, 23, 30, tzinfo=timezone.utc)

print("naïf,  tz=Paris  :", bcal.session_of(naif, tz="Europe/Paris"), " <- lu littéralement")
print("aware, tz=Paris  :", bcal.session_of(aware, tz="Europe/Paris"), " <- converti")

naïf,  tz=Paris  : 2026-07-31  <- lu littéralement
aware, tz=Paris  : 2026-08-01  <- converti


## 4. Les offsets préservent l'heure murale

Un décalage ne touche que la partie date. À travers un changement d'heure, cela veut dire
que +1 jour ouvré vaut +23 h ou +25 h en temps absolu — et c'est **voulu**.

In [8]:
PARIS = ZoneInfo("Europe/Paris")
avant = datetime(2026, 3, 27, 9, 0, tzinfo=PARIS)          # vendredi, avant le passage
apres = paris.offset(avant, 1)                              # -> lundi

ecoule = apres.astimezone(timezone.utc) - avant.astimezone(timezone.utc)
print("avant   :", avant)
print("+1 ouvré:", apres)
print("horloge murale identique :", avant.hour == apres.hour)
print("temps réellement écoulé  :", ecoule, "au lieu de 72 h")

avant   : 2026-03-27 09:00:00+01:00
+1 ouvré: 2026-03-30 09:00:00+02:00
horloge murale identique : True
temps réellement écoulé  : 2 days, 23:00:00 au lieu de 72 h


## 5. Sessions

Un jour calendaire est l'intervalle `[session_start, session_start + 24h)` exprimé dans le
fuseau du calendrier. Minuit local pour la plupart, `00:00` UTC pour le crypto,
`17:00` New York pour le FX.

In [9]:
fx = Calendar("fx", tz="America/New_York", session_start=pd.Timestamp("17:00").time())

matin = pd.Timestamp("2026-07-31 09:00", tz="America/New_York")
soir = pd.Timestamp("2026-07-31 18:00", tz="America/New_York")

print("session_start = 17:00 New York")
print(f"  {matin}  -> session {bcal.session_of(matin, cal=fx)}   (ouverte la veille au soir)")
print(f"  {soir}  -> session {bcal.session_of(soir, cal=fx)}")

session_start = 17:00 New York
  2026-07-31 09:00:00-04:00  -> session 2026-07-30   (ouverte la veille au soir)
  2026-07-31 18:00:00-04:00  -> session 2026-07-31


`session_bounds` donne l'intervalle UTC semi-ouvert que couvre un jour :

In [10]:
xpar = bcal.get("XPAR")
pd.DataFrame(
    [
        {
            "jour": jour,
            "début (UTC)": str(xpar.session_bounds(jour)[0]),
            "fin (UTC)": str(xpar.session_bounds(jour)[1]),
            "durée": str(xpar.session_bounds(jour)[1] - xpar.session_bounds(jour)[0]),
        }
        for jour in ("2026-03-27", "2026-03-29", "2026-07-31", "2026-10-25")
    ]
).set_index("jour")

,début (UTC),fin (UTC),durée
jour,,,
2026-03-27,2026-03-26 23:00:00+00:00,2026-03-27 23:00:00+00:00,1 days 00:00:00
2026-03-29,2026-03-28 23:00:00+00:00,2026-03-29 22:00:00+00:00,0 days 23:00:00
2026-07-31,2026-07-30 22:00:00+00:00,2026-07-31 22:00:00+00:00,1 days 00:00:00
2026-10-25,2026-10-24 22:00:00+00:00,2026-10-25 23:00:00+00:00,1 days 01:00:00


Une session fait réellement 23 h ou 25 h autour d'un changement d'heure. Ce n'est pas un
défaut à normaliser : c'est le code qui suppose 24 h que cette fonction existe pour
corriger.

Les deux fonctions sont cohérentes par construction — tout instant tombe dans les bornes
du jour que `session_of` lui attribue :

In [11]:
ok = True
for calendrier in (utc, paris, fx, bcal.get("crypto:24x7")):
    for heure in range(0, 24, 3):
        moment = pd.Timestamp(f"2026-07-31 {heure:02d}:17", tz="UTC")
        jour = bcal.session_of(moment, cal=calendrier)
        debut, fin = calendrier.session_bounds(jour)
        ok &= bool(debut <= moment < fin)
print("cohérence session_of / session_bounds sur 4 calendriers × 8 heures :", ok)

cohérence session_of / session_bounds sur 4 calendriers × 8 heures : True


## 6. `grid` : le resample mal ancré

C'est le piège classique. Une grille de 4 h construite depuis minuit UTC coupe une séance
de Paris ou de Tokyo aux mauvais endroits.

In [12]:
naif_pandas = pd.date_range("2026-07-31 00:00", periods=6, freq="4h", tz="UTC")
aligne = xpar.grid("2026-07-31", "2026-07-31", "4h")

pd.DataFrame(
    {
        "pandas depuis minuit UTC": naif_pandas.tz_convert("Europe/Paris").strftime("%m-%d %H:%M"),
        "grid ancré sur la séance": aligne.tz_convert("Europe/Paris").strftime("%m-%d %H:%M"),
    }
)

,pandas depuis minuit UTC,grid ancré sur la séance
0,07-31 02:00,07-31 00:00
1,07-31 06:00,07-31 04:00
2,07-31 10:00,07-31 08:00
3,07-31 14:00,07-31 12:00
4,07-31 18:00,07-31 16:00
5,07-31 22:00,07-31 20:00


La colonne de gauche démarre à 02:00 heure de Paris : les barres sont décalées de deux
heures par rapport à la journée réelle. La colonne de droite démarre à minuit local.

`grid` ne couvre que les **séances**, donc pas de barre le week-end pour une bourse, et une
grille continue pour le crypto :

In [13]:
print("XPAR, samedi + dimanche :", len(xpar.grid("2026-08-01", "2026-08-02", "6h")), "points")
print("crypto, samedi + dimanche:", len(bcal.get("crypto:24x7").grid("2026-08-01", "2026-08-02", "6h")), "points")

XPAR, samedi + dimanche : 0 points
crypto, samedi + dimanche: 8 points


Chaque séance est remplie indépendamment, donc un changement d'heure raccourcit **cette**
séance-là sans décaler tous les points suivants :

In [14]:
autour = xpar.grid("2026-03-27", "2026-03-31", "6h")
locale = autour.tz_convert("Europe/Paris")
pd.DataFrame({"UTC": autour.strftime("%m-%d %H:%M"), "Paris": locale.strftime("%m-%d %H:%M")})

,UTC,Paris
0,03-26 23:00,03-27 00:00
1,03-27 05:00,03-27 06:00
2,03-27 11:00,03-27 12:00
3,03-27 17:00,03-27 18:00
4,03-29 22:00,03-30 00:00
5,03-30 04:00,03-30 06:00
6,03-30 10:00,03-30 12:00
7,03-30 16:00,03-30 18:00
8,03-30 22:00,03-31 00:00
9,03-31 04:00,03-31 06:00


## 7. `at_times` : croiser des jours et des heures

Le compagnon des récurrences : on génère les jours, on y accroche les heures auxquelles un
process tourne réellement.

In [15]:
fixings = bcal.at_times(bcal.imm_dates("2026-01-01", "2026-12-31"), ["08:00", "16:00"])
pd.DataFrame({"UTC": fixings.strftime("%Y-%m-%d %H:%M%z")})

,UTC
0,2026-03-18 08:00+0000
1,2026-03-18 16:00+0000
2,2026-06-17 08:00+0000
3,2026-06-17 16:00+0000
4,2026-09-16 08:00+0000
5,2026-09-16 16:00+0000
6,2026-12-16 08:00+0000
7,2026-12-16 16:00+0000


In [16]:
# Dans un autre fuseau, avec des secondes.
bcal.at_times(["2026-07-31"], ["09:30:15", "17:00"], tz="Europe/Paris").strftime("%Y-%m-%d %H:%M:%S %Z").tolist()

['2026-07-31 09:30:15 CEST', '2026-07-31 17:00:00 CEST']

## 8. Ce qui est délibérément absent

Pas de `is_open`, pas de `next_open`, pas de pauses déjeuner, pas de clôtures anticipées.

Un `is_open()` qui renverrait `is_bday()` serait **faux** pour toute bourse ayant des
horaires de cotation — et ça se découvrirait de la pire des façons. La distinction est
écrite noir sur blanc sous forme de deux protocoles, `DayCalendar` (satisfait aujourd'hui)
et `SessionCalendar` (satisfait par rien), pour qu'elle survive au prochain qui touchera
ce code.

In [17]:
[nom for nom in ("is_open", "next_open", "next_close", "trading_minutes")
 if hasattr(bcal.get("XNYS"), nom)] or "aucune de ces méthodes n'existe"

"aucune de ces méthodes n'existe"

## Récapitulatif

| Appel | Rôle |
|---|---|
| `session_of(ts, cal=, tz=)` | à quel jour calendaire appartient cet instant |
| `session_bounds(jour, cal=)` | l'intervalle UTC semi-ouvert du jour |
| `cal.grid(a, b, "4h")` | grille ancrée sur `session_start`, pas sur minuit UTC |
| `at_times(jours, heures, tz=)` | croiser une récurrence avec des heures |
| `config.default_tz` | l'échappatoire globale, désactivée par défaut |

**Suite :** [06 — Snapshots, provenance et overrides](06-snapshots-et-overrides.ipynb)